# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 06 · Retrieval design

This notebook explains the **retrieval strategy** designed for the GitHub RAG system.

#### **Notebook objectives**

* Present the **adaptive retrieval strategy** and the rationale behind its design.
* Explain how each retrieval strategy works and when it is applied.
* Describe the main **architecture and design decisions**, maintaining a clear separation of responsibilities between the retrieval layer and the vector database.

## 1. Retrieval in the RAG pipeline

Retrieval is a fundamental component of a **Retrieval-Augmented Generation (RAG)** system. Its purpose is to identify and retrieve the repository documents that are most relevant to a user's query before passing them to the language model.

In this project, the knowledge base consists of documents extracted from a GitHub repository. These documents represent source-code files, documentation, notebooks, configuration files, and other repository contents that have been processed, chunked, embedded, and stored in ChromaDB.

The retrieval stage establishes the connection between the user's natural-language query and the relevant parts of the repository. Given a query, the system searches the vector database using the embeddings generated during the ingestion pipeline and returns a set of relevant chunks that will subsequently form the context provided to the LLM.

The quality of this stage is particularly important because the language model can only generate a reliable answer if the retrieved context contains the information required to answer the query. Retrieving too little relevant information may lead to incomplete answers, whereas retrieving too much or redundant information unnecessarily increases the context size and can introduce irrelevant information.

Therefore, retrieval is not treated as a single fixed operation in this project. Instead, the retrieval strategy is adapted to the structure of the user's query.

## 2. Motivation for an adaptive retrieval strategy

A repository-level RAG system can receive queries with very different levels of specificity.

For example, a user may explicitly identify a single file:

> "Explain what is uncertainty filtering implemented in `AL_methods.py.`"

Alternatively, the user may request a comparison between multiple files:

> "Compare how active learning is evaluated in `AL_performance.py` and `AL_scalability.py`."

Finally, the user may ask a general question without knowing where the relevant information is located:

> "How is active learning implemented in this repository?"

These queries impose different requirements on the retrieval process. A fixed retrieval strategy would simplify the implementation, but it would not take advantage of information already provided by the user.

For example, the first query defines a narrow search space. The second explicitly requires information from multiple sources. The third requires repository-level exploration.

Using the same strategy and the same number of retrieved chunks for all three cases would therefore represent an unnecessary compromise.

For this reason, the `Retriever` component implements an adaptive retrieval policy based on the number of repository files referenced in the query:

```text
                    User query
                        │
                        ▼
              Detect file references
                        │
          ┌─────────────┼─────────────┐
          │             │             │
          ▼             ▼             ▼
       1 file        > 1 file       0 files
          │             │             │
          ▼             ▼             ▼
     Similarity        MMR           MMR
       search        + filter     whole repository
          │             │             │
         k=5          k=10          k=5
          │             │             │
          └─────────────┼─────────────┘
                        ▼
                Retrieved context
                        │
                        ▼
                       LLM
```

The resulting design prioritizes:

* **Relevance** when the user has clearly defined the retrieval scope.
* **Diversity and broader coverage** when information from multiple files may be required.
* **Repository exploration** when the user has not identified the location of the relevant information.
* **Context control** by keeping the number of retrieved chunks bounded.

The operation of each retrieval strategy is presented in detail below.

### 2.1 Single-file queries: semantic similarity search

When the query references a single repository file, the retrieval context is already well defined.

For example:

> "Explain what is uncertainty filtering implemented in `AL_methods.py`."

The system first identifies the referenced repository path and uses it as a metadata filter:

```python
{
    "file_path": "scripts/AL_methods.py"
}
```

The retrieval operation is therefore restricted to chunks belonging to that file.

In this scenario, standard semantic similarity search is sufficient. The objective is to identify the chunks within the specified file that are most semantically relevant to the user's question.

The main reason for not using MMR in this case is that the search space is already constrained to a single file. Introducing a diversity-oriented retrieval strategy provides less additional value because the user has explicitly defined the contextual scope.

Therefore, the strategy prioritizes **relevance over diversity**.

### 2.2. Multiple-file queries: MMR retrieval

A different situation occurs when the query references multiple files.

For example:

> "Compare how active learning is evaluated in `AL_performance.py` and `AL_scalability.py`."

This type of query requires information from several parts of the repository. Simply retrieving the globally most similar chunks from the selected files may result in several highly similar chunks originating from the same file or from the same section of code.

To mitigate this effect, the system uses **Maximal Marginal Relevance (MMR)**.

MMR balances two objectives:

1. Relevance to the original query.
2. Diversity with respect to documents that have already been selected.

Conceptually, the selection criterion can be expressed as:

$$
MMR(d) =
\lambda \cdot Sim(d,q)
-
(1-\lambda) \cdot
\max_{d' \in S} Sim(d,d')
$$

where:

* \(d\) is a candidate document,
* \(q\) is the user query,
* \(S\) is the set of documents already selected,
* \(Sim(d,q)\) measures relevance to the query,
* \(Sim(d,d')\) measures similarity to previously selected documents,
* \(\lambda\) controls the trade-off between relevance and diversity.

In this project, MMR is particularly useful for multi-file queries because the retrieved context should cover the different files involved rather than concentrating excessively on a single highly similar region.

The search is therefore restricted to the referenced files using a metadata filter:

```python
{
    "file_path": {
        "$in": [
            "scripts/AL_performance.py",
            "scripts/AL_scalability.py"
        ]
    }
}
```

The number of retrieved chunks is increased to $10$. This provides a broader context for queries that require information from multiple files while maintaining a bounded context size.

This strategy is particularly appropriate for comparative queries, where the answer may depend on understanding how different components of the repository interact or differ.

### 2.3 Queries without file references: repository-level MMR

The most open-ended case occurs when the user does not explicitly identify any repository file.

For example:

> "How is active learning implemented in this repository?"

Here, the user is interested in a repository-level concept but does not know where the relevant implementation is located.

Restricting the search to a particular file is therefore not possible. A global semantic search could retrieve several highly similar chunks from the same file or from the same implementation area, potentially reducing the coverage of the retrieved context.

MMR is used to address this problem.

The search is performed over the complete vector database, allowing the retrieval process to explore different parts of the repository while favouring relevant and non-redundant information.

However, unlike the multi-file case, only 5 chunks are retrieved.

This is an intentional design decision. When the user has not specified a contextual scope, increasing `k` can unnecessarily expand the amount of information passed to the language model. A smaller retrieval set reduces the context size while still providing several diverse candidates from which the relevant repository components can be identified.

This provides a compromise between **repository coverage and context efficiency**.

### 2.4 Design summary

The retrieval strategy implemented in this project is therefore not based solely on vector similarity. It combines semantic retrieval, metadata filtering, and diversity-aware retrieval according to the level of specificity expressed by the user.

The key design decisions are:

1. **Single-file reference → semantic search (`k=5`)**
   The retrieval scope is already constrained to one file, making relevance the primary objective.

2. **Multiple-file references → filtered MMR (`k=10`)**
   The query potentially requires information from several files. MMR reduces redundancy and provides broader coverage of the specified context.

3. **No file reference → global MMR (`k=5`)**
   The user has not identified where the relevant information is located. MMR provides diverse repository-level candidates while keeping the retrieved context limited.

This adaptive strategy aims to provide the language model with a context that is **relevant, sufficiently diverse, and appropriately sized for the user's query**, rather than maximizing the number of retrieved documents indiscriminately.

## 3. Separation of retrieval responsibilities

The retrieval architecture separates the decision-making logic from the vector-database implementation.

The `Retriever` (located in *src/github_rag/retrieval/retriever.py*) component is responsible for interpreting the query and selecting the appropriate retrieval strategy:

```text
Retriever
├── Detect referenced files
├── Select retrieval strategy
├── Determine k
├── Build metadata filters
└── Prepare the query
```

The `ChromaStore` (located in *src/github_rag/vectordatabase/chroma_store.py*) component is responsible for interacting with ChromaDB:

```text
ChromaStore
├── Semantic similarity search
└── Maximal Marginal Relevance search
```

This separation avoids coupling the application-level retrieval policy to the underlying vector database.

For example, the `Retriever` does not need to know how ChromaDB internally performs similarity search or MMR. It only specifies the required operation, query, number of results, and optional metadata filter.

This design also makes the retrieval policy easier to test independently from the vector database.